# KNN Classification of Cyberbullying Tweets (all 6 classes) with BERT embeddings

A **supervised** classifier: BERT turns each tweet into a vector, then **K-Nearest-
Neighbors** predicts one of the 6 categories. Because KNN is supervised, train/test
split and accuracy are the natural way to evaluate it.

Pipeline: load → clean & dedupe → encode labels → **train/test split** → BERT embeddings
→ **tune k by cross-validation** → train → evaluate on train & test → error analysis.


In [ ]:
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from scipy import stats

sns.set_style('whitegrid')
RANDOM_STATE = 23

## Step 1 — Load & inspect

In [ ]:
df = pd.read_csv('cyberbullying_tweets.csv')
print('Raw shape:', df.shape)
df['cyberbullying_type'].value_counts()

Raw shape: (47692, 2)


cyberbullying_type
religion               7998
age                    7992
gender                 7973
ethnicity              7961
not_cyberbullying      7945
other_cyberbullying    7823
Name: count, dtype: int64

In [ ]:
plt.figure(figsize=(8, 4))
order = df['cyberbullying_type'].value_counts().index
sns.countplot(y='cyberbullying_type', data=df, order=order, palette='viridis')
plt.title('Full class distribution (all 6 classes)')
plt.xlabel('count'); plt.ylabel('')
plt.tight_layout(); plt.show()

C:\Users\Awais\AppData\Local\Temp\ipykernel_17608\3937048739.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(y='cyberbullying_type', data=df, order=order, palette='viridis')
C:\Users\Awais\AppData\Local\Temp\ipykernel_17608\3937048739.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Step 2 — Clean & deduplicate

Remove duplicate rows, then light cleaning for BERT (strip URLs & @mentions, keep
casing/punctuation).

In [ ]:
before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - df.shape[0]} duplicate rows -> {df.shape[0]} rows')

def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['tweet_text'].apply(clean_text)
df[['tweet_text', 'clean_text']].head()

Removed 36 duplicate rows -> 47656 rows


,tweet_text,clean_text
0,"In other words #katandandre, your food was cra...","In other words #katandandre, your food was cra..."
1,Why is #aussietv so white? #MKR #theblock #ImA...,Why is #aussietv so white? #MKR #theblock #ImA...
2,@XochitlSuckkks a classy whore? Or more red ve...,a classy whore? Or more red velvet cupcakes?
3,"@Jason_Gio meh. :P thanks for the heads up, b...","meh. :P thanks for the heads up, but not too c..."
4,@RudhoeEnglish This is an ISIS account pretend...,This is an ISIS account pretending to be a Kur...


## Step 3 — Encode labels (class name -> integer 0..5)

In [ ]:
classes = sorted(df['cyberbullying_type'].unique())
cls2int = {c: i for i, c in enumerate(classes)}
int2cls = {i: c for c, i in cls2int.items()}
df['label'] = df['cyberbullying_type'].map(cls2int)

pd.DataFrame({'class': classes, 'integer': [cls2int[c] for c in classes]})

,class,integer
0,age,0
1,ethnicity,1
2,gender,2
3,not_cyberbullying,3
4,other_cyberbullying,4
5,religion,5


## Step 4 — Train/test split (stratified 80/20)

The test set is locked away and never used until the very end. Stratifying keeps the
6 classes balanced in both sets. We split the *texts* first, then embed each side
separately (BERT is frozen, so no train->test leakage).

In [ ]:
text_train, text_test, y_train, y_test = train_test_split(
    df['clean_text'].values, df['label'].values,
    test_size=0.2, stratify=df['label'].values, random_state=RANDOM_STATE)
print('train:', len(text_train), ' test:', len(text_test))

# Confirm stratification
cmp = pd.DataFrame({
    'train': pd.Series(y_train).map(int2cls).value_counts(),
    'test':  pd.Series(y_test).map(int2cls).value_counts(),
})
cmp.plot(kind='bar', figsize=(9, 4), color=['#4c72b0', '#dd8452'])
plt.title('Class counts: train vs test (stratified)')
plt.ylabel('count'); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()
cmp

train: 38124  test: 9532


C:\Users\Awais\AppData\Local\Temp\ipykernel_17608\3223444480.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


,train,test
religion,6397,1600
age,6394,1598
ethnicity,6367,1592
gender,6358,1590
not_cyberbullying,6350,1587
other_cyberbullying,6258,1565


## Step 5 — BERT embeddings

Each tweet -> a 384-dim vector from `all-MiniLM-L6-v2`. We cache the embeddings to disk
so re-running the notebook is fast. The first run takes ~5 min on CPU.

In [ ]:
cache = 'bert_embeddings.npz'
if os.path.exists(cache):
    d = np.load(cache)
    X_train, X_test = d['X_train'], d['X_test']
    print('Loaded cached embeddings.')
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    X_train = np.asarray(model.encode(text_train.tolist(), batch_size=64,
                         normalize_embeddings=True, show_progress_bar=True), dtype=np.float32)
    X_test = np.asarray(model.encode(text_test.tolist(), batch_size=64,
                        normalize_embeddings=True, show_progress_bar=True), dtype=np.float32)
    np.savez(cache, X_train=X_train, X_test=X_test)

print('X_train:', X_train.shape, ' X_test:', X_test.shape)

Loaded cached embeddings.
X_train: (38124, 384)  X_test: (9532, 384)


## Step 5b — See the embedding space (PCA & t-SNE, colored by class)

2D projections are for the eyes only — KNN runs on the full 384-dim vectors. If the
classes form visible regions here, KNN has something to work with.

In [ ]:
# PCA view of the training embeddings
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_train)

plt.figure(figsize=(8, 6))
for i in range(len(classes)):
    m = y_train == i
    plt.scatter(pca_2d[m, 0], pca_2d[m, 1], s=5, alpha=0.4, label=int2cls[i])
plt.title('PCA of BERT embeddings (train) by class')
plt.legend(markerscale=2, loc='best')
plt.tight_layout(); plt.show()

C:\Users\Awais\AppData\Local\Temp\ipykernel_17608\1620855039.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [ ]:
# t-SNE on a random sample (t-SNE is slow)
samp = np.random.RandomState(RANDOM_STATE).choice(len(X_train), size=5000, replace=False)
tsne_2d = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30,
               init='pca').fit_transform(X_train[samp])

plt.figure(figsize=(8, 6))
for i in range(len(classes)):
    m = y_train[samp] == i
    plt.scatter(tsne_2d[m, 0], tsne_2d[m, 1], s=7, alpha=0.5, label=int2cls[i])
plt.title('t-SNE of BERT embeddings (5k sample) by class')
plt.legend(markerscale=2, loc='best')
plt.tight_layout(); plt.show()

C:\Users\Awais\AppData\Local\Temp\ipykernel_17608\1777057484.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Step 6 — Tune k by cross-validation

For each candidate k (1..30) we measure accuracy with 5-fold CV **on the training set
only**. To stay fast, we find the 30 nearest neighbors once per fold and reuse them for
every k (take the first k and let them vote).

In [ ]:
max_k = 30
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = np.zeros(max_k)

for tr, va in skf.split(X_train, y_train):
    nn = NearestNeighbors(n_neighbors=max_k, metric='euclidean').fit(X_train[tr])
    neigh_idx = nn.kneighbors(X_train[va], return_distance=False)   # (n_va, max_k)
    neigh_lab = y_train[tr][neigh_idx]                              # neighbor labels
    for k in range(1, max_k + 1):
        pred = stats.mode(neigh_lab[:, :k], axis=1, keepdims=False).mode
        cv_scores[k - 1] += np.mean(pred == y_train[va])
cv_scores /= skf.get_n_splits()

best_k = int(np.argmax(cv_scores) + 1)
print(f'Best k = {best_k}  (CV accuracy {cv_scores[best_k-1]:.4f})')

Best k = 16  (CV accuracy 0.8074)


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, max_k + 1), cv_scores, marker='o')
plt.axvline(best_k, color='red', ls='--', label=f'best k = {best_k}')
plt.scatter([best_k], [cv_scores[best_k-1]], color='red', zorder=5)
plt.title('Cross-validated accuracy vs k')
plt.xlabel('k (number of neighbors)'); plt.ylabel('mean CV accuracy')
plt.legend(); plt.tight_layout(); plt.show()

C:\Users\Awais\AppData\Local\Temp\ipykernel_17608\2234819334.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Table: top 5 values of k
cv_table = pd.DataFrame({'k': range(1, max_k + 1), 'cv_accuracy': cv_scores})
cv_table.sort_values('cv_accuracy', ascending=False).head(5).reset_index(drop=True)

,k,cv_accuracy
0,16,0.807392
1,15,0.807156
2,19,0.806631
3,20,0.806526
4,18,0.806343


## Step 7 — Train the final model with best_k

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
knn.fit(X_train, y_train)   # KNN 'training' = storing the train vectors + labels
print('Trained KNN with k =', best_k)

Trained KNN with k = 16


## Step 8 — Evaluate on TRAIN (overfitting check)

Train accuracy alone is not the real result — with small k it is optimistic because a
tweet can find near-copies of itself. We compare it to test accuracy next.

In [ ]:
train_pred = knn.predict(X_train)
train_acc = accuracy_score(y_train, train_pred)
print(f'TRAIN accuracy: {train_acc:.4f}')

cm_train = confusion_matrix(y_train, train_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title(f'Confusion matrix — TRAIN (k={best_k})')
plt.ylabel('true'); plt.xlabel('predicted')
plt.xticks(rotation=40, ha='right'); plt.tight_layout(); plt.show()

TRAIN accuracy: 0.8294


C:\Users\Awais\AppData\Local\Temp\ipykernel_17608\1087431856.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xticks(rotation=40, ha='right'); plt.tight_layout(); plt.show()


## Step 9 — Evaluate on TEST (the real result)

In [ ]:
test_pred = knn.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)
macro_f1 = f1_score(y_test, test_pred, average='macro')
print(f'TEST accuracy : {test_acc:.4f}')
print(f'TEST macro-F1 : {macro_f1:.4f}')

TEST accuracy : 0.8121
TEST macro-F1 : 0.8035


In [ ]:
# Full per-class report as a table
report = classification_report(y_test, test_pred, target_names=classes,
                               output_dict=True)
pd.DataFrame(report).T.round(3)

,precision,recall,f1-score,support
age,0.850,0.975,0.908,1598.000
ethnicity,0.910,0.943,0.926,1592.000
gender,0.824,0.856,0.840,1590.000
not_cyberbullying,0.669,0.505,0.576,1587.000
other_cyberbullying,0.647,0.617,0.632,1565.000
religion,0.909,0.971,0.939,1600.000
accuracy,0.812,0.812,0.812,0.812
macro avg,0.802,0.811,0.803,9532.000
weighted avg,0.802,0.812,0.804,9532.000


In [ ]:
# Confusion matrix (raw counts) and normalized (row %) side by side
cm = confusion_matrix(y_test, test_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title(f'Confusion matrix — TEST counts (k={best_k})')
axes[0].set_ylabel('true'); axes[0].set_xlabel('predicted')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('Confusion matrix — TEST normalized (row %)')
axes[1].set_ylabel('true'); axes[1].set_xlabel('predicted')

for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')
plt.tight_layout(); plt.show()

C:\Users\Awais\AppData\Local\Temp\ipykernel_17608\493126930.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [ ]:
# Per-class F1 bar chart
f1_per = f1_score(y_test, test_pred, average=None)
plt.figure(figsize=(9, 4))
bars = plt.bar(classes, f1_per, color=sns.color_palette('viridis', len(classes)))
plt.title('Per-class F1 on TEST'); plt.ylabel('F1'); plt.ylim(0, 1)
plt.xticks(rotation=40, ha='right')
for b, v in zip(bars, f1_per):
    plt.text(b.get_x() + b.get_width()/2, v, f'{v:.2f}', ha='center', va='bottom')
plt.tight_layout(); plt.show()

C:\Users\Awais\AppData\Local\Temp\ipykernel_17608\2547562279.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Step 10 — Error analysis: a sample of misclassified test tweets

In [ ]:
wrong = np.where(test_pred != y_test)[0]
rng = np.random.RandomState(RANDOM_STATE)
pick = rng.choice(wrong, size=min(12, len(wrong)), replace=False)

err = pd.DataFrame({
    'tweet': [text_test[i][:90] for i in pick],
    'true': [int2cls[y_test[i]] for i in pick],
    'predicted': [int2cls[test_pred[i]] for i in pick],
})
err

,tweet,true,predicted
0,"i think in the past, there were more grey area...",other_cyberbullying,ethnicity
1,"RT : So, if you thought that this whole Wadwha...",other_cyberbullying,gender
2,RT : Nikki has massive #armpitvaginas #mkr,gender,not_cyberbullying
3,Dev week matches my outfit!,other_cyberbullying,not_cyberbullying
4,twitter allows 13 yr olds. 13 yr olds don't ge...,not_cyberbullying,other_cyberbullying
5,Yo /baph my address was never even that apartm...,not_cyberbullying,ethnicity
6,Why can't you tell?,other_cyberbullying,not_cyberbullying
7,Thinkin Bout My Righthand Im Finna Text Her :/,not_cyberbullying,ethnicity
8,Who said that?,gender,other_cyberbullying
9,Whiny manbabies.,other_cyberbullying,ethnicity


## Step 11 — Summary

In [ ]:
summary = pd.DataFrame({
    'metric': ['best k', 'train accuracy', 'test accuracy', 'test macro-F1'],
    'value':  [best_k, round(train_acc, 4), round(test_acc, 4), round(macro_f1, 4)],
})
summary

,metric,value
0,best k,16.0000
1,train accuracy,0.8294
2,test accuracy,0.8121
3,test macro-F1,0.8035
